In [ ]:
import random

from distributed_processing.client import Client
from distributed_processing.redis_connector import RedisConnector


In [ ]:
REDIS_HOST = "localhost"
REDIS_PORT = 6379
REDIS_DB = 0
NAMESPACE = "montecarlos"

In [ ]:
redis_connector = RedisConnector(redis_host=REDIS_HOST, redis_port=REDIS_PORT, 
                                 redis_db=REDIS_DB, namespace=NAMESPACE)

client = Client(redis_connector, check_registry="cache")

In [ ]:
import datetime

NSIMULATIONS = 1000000

V=0.20 # Volatilidad
R=0.04 # Interés instantáneo (ln(1+r)?)
COUPON_BARRIER=0.8
KICKOUT_BARRIER=1.1
PROTECTION_BARRIER=0.6
COUPON_RATE=0.088

obs_dates = [datetime.datetime(2012, 7, 4, 0, 0),
             datetime.datetime(2013, 7, 5, 0, 0),
             datetime.datetime(2014, 7, 7, 0, 0),
             datetime.datetime(2015, 7, 6, 0, 0),
             datetime.datetime(2016, 7, 5, 0, 0),
             datetime.datetime(2017, 7, 5, 0, 0),
             datetime.datetime(2018, 7, 5, 0, 0)]

dates = [int((t-obs_dates[0]).days) for t in obs_dates]

In [ ]:
# Versión basada en RandomState
# RandomState se ha quedado anticuado y utilza el generador MT19937
# que es más lento que el generador ahora por defecto en numpy PGC64 


nchunks = 32
nsims = float(NSIMULATIONS) / float(nchunks)

def chunks(nsims, nchunks):
    chunks = [int(nsims)] * nchunks
    diff = NSIMULATIONS - int(nsims) * nchunks
    if diff>0:
        for i in range(diff):
            chunks[i] += 1
    return chunks


def args_kwargs(*args, **kwargs):
    return list(args), kwargs



def args_kwargs1(nsimulations):
    return args_kwargs(nsimulations=nsimulations,
                        v=V,
                        r=R, 
                        coupon_barrier=COUPON_BARRIER,
                        kickout_barrier=KICKOUT_BARRIER,
                        protection_barrier=PROTECTION_BARRIER,
                        coupon_rate=COUPON_RATE,
                        dates=dates)

all_args = [args_kwargs1(x) for x in chunks(int(nsims), nchunks)]


def distributed_montecarlo():
    sim = [client.rpc_async("mc_autocall_mp", x[0], x[1]) for x in all_args]
    sumas, sizes = list(zip(*[x.get() for x in sim]))
    
    return sum(sumas)/sum(sizes)
    

In [ ]:
%%timeit
distributed_montecarlo()

In [ ]:
distributed_montecarlo()

In [ ]:
sim = [client.rpc_async("mc_autocall_mp", x[0], x[1]) for x in all_args]
sim

In [ ]:
[x.get() for x in sim]

In [ ]:
sumas, sizes = list(zip(*[x.get() for x in sim]))
sum(sumas)/sum(sizes)

In [ ]:
# Versión basada en PGC64. Hay que pasar a la función la semilla y el número de salto.
# Basado en https://numpy.org/doc/1.18/reference/random/parallel.html

import secrets


def args_kwargs2(x):
    seed = int(x[0])
    jump = int(x[1])
    nsim = int(x[2])
    return args_kwargs(rndg=(seed, jump),
                        nsimulations=nsim,
                        v=V,
                        r=R, 
                        coupon_barrier=COUPON_BARRIER,
                        kickout_barrier=KICKOUT_BARRIER,
                        protection_barrier=PROTECTION_BARRIER,
                        coupon_rate=COUPON_RATE,
                        dates=dates)



seed = secrets.randbits(128)

rnd_args = [x for x in zip([seed]*nchunks, range(nchunks),  chunks(int(nsims), nchunks))]

all_args = [args_kwargs2(x) for x in rnd_args]

def distributed_montecarlo2():
    sim = [client.rpc_async("mc_autocall_mp2", x[0], x[1]) for x in all_args]
    sumas, sizes = list(zip(*[x.get() for x in sim]))
    
    return sum(sumas)/sum(sizes)
    

In [ ]:
%%timeit
distributed_montecarlo2()

In [ ]:
distributed_montecarlo2()

In [ ]:
# Versión basada en PGC64. Hay que pasar a la función la semilla y el número de salto.
# Basado en https://numpy.org/doc/1.18/reference/random/parallel.html

import secrets

nchunks = 32
nsims = float(NSIMULATIONS) / float(nchunks)

def args_kwargs2(x):
    seed = int(x[0])
    jump = int(x[1])
    nsim = int(x[2])
    return args_kwargs(rndg=(seed, jump),
                        nsimulations=nsim,
                        v=V,
                        r=R, 
                        coupon_barrier=COUPON_BARRIER,
                        kickout_barrier=KICKOUT_BARRIER,
                        protection_barrier=PROTECTION_BARRIER,
                        coupon_rate=COUPON_RATE,
                        dates=dates)



seed = secrets.randbits(128)

rnd_args = [x for x in zip([seed]*nchunks, range(nchunks),  chunks(int(nsims), nchunks))]

all_args = [args_kwargs2(x) for x in rnd_args]

def distributed_empty_montecarlo():
    sim = [client.rpc_async("mc_autocall_empty", x[0], x[1]) for x in all_args]
    sumas, sizes = list(zip(*[x.get() for x in sim]))
    
    return sum(sumas)/sum(sizes)
    

In [ ]:
%%timeit
distributed_empty_montecarlo()